# Random Forest Example: Predicting Wide Receiver Draft Round

If you haven't seen yet, check out the decision tree example before continuing on in this notebook.

In this notebook we will attempt to predict wide receiver draft round based on the same stats and accomplishments as the decision tree example. This will allow us to compare the effectiveness of a single decision tree model and a random forest model.

We hypothesize that the random forest will perform better, but is also likely limited due to the limited scope of our analysis. The NFL Draft isn't as straightforward as players in higher rounds do better and that's what makes the NFL so fascinating. Players develop and change as they grow older and experience the NFL culture, making the league and its storylines so interesting. However, that's what makes these models so difficult.

In [1]:
# Import necessary libraries
import sys
import os

# Send Python to the project root so we can import our library
project_root = "/Users/maxbasurto/Documents/Rice Classes/CMOR 438/CMOR438-Project/src"
sys.path.append(project_root)

# Import our ML library and other necessary libraries
import rice_ml
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# Import the dataset
data_path = "/Users/maxbasurto/Documents/Rice Classes/CMOR 438/CMOR438-Project/data/nfl_draft_data.csv"
data = pd.read_csv(data_path)

Now that we have the data and package imported, we can continue as we did in the decision tree notebook.

In [3]:
# Filter the dataset to include only the relevant columns
data = data[["season", "pfr_player_name", "round", "position", "receptions","rec_yards", "rec_tds", "hof", "allpro", "probowls"]]

# Only want Wide Receivers
data = data[data["position"] == "WR"]

# Only want players drafted this century
data = data[data["season"] >= 2000]
# Now we don't need the season column
data = data.drop(columns=["season"])

# Drop rows with missing values
data = data.dropna()

# View the first few rows of the dataset
print(data.head())

       pfr_player_name  round position  receptions  rec_yards  rec_tds    hof  \
6029     Peter Warrick      1       WR       275.0     2991.0     18.0  False   
6033   Plaxico Burress      1       WR       553.0     8499.0     64.0  False   
6035     Travis Taylor      1       WR       312.0     4017.0     22.0  False   
6046  Sylvester Morris      1       WR        48.0      678.0      3.0  False   
6054     R. Jay Soward      1       WR        14.0      154.0      1.0  False   

      allpro  probowls  
6029       0         0  
6033       0         0  
6035       0         0  
6046       0         0  
6054       0         0  


As explained in the decision tree notebook, we clean the data to better reflect the current state of the NFL and select only the necessary features and players.

In [38]:
# Standardize the data using our StandardScaler class from our library
scaler = rice_ml.StandardScaler()
# Must convert hof to integer values (1 for yes, 0 for no) before scaling
data["hof"] = data["hof"].astype(int)
# Now can scale the features
scaled_data = scaler.fit_transform(data[["receptions", "rec_yards", "rec_tds", "hof", "allpro", "probowls"]].values)

# Combine scaled features with the target variable (round) into a new DataFrame
scaled_data = pd.DataFrame(scaled_data, columns=["receptions", "rec_yards", "rec_tds", "hof", "allpro", "probowls"])
scaled_data["round"] = data["round"].values

# Bring player names back for later analysis of predictions
scaled_data["pfr_player_name"] = data["pfr_player_name"].values

# Create a 80/20 train/test split with random shuffling
scaled_data = scaled_data.sample(frac=1, random_state=42).reset_index(drop=True)
train_size = int(0.8 * len(scaled_data))
train_data = scaled_data.iloc[:train_size]
test_data = scaled_data.iloc[train_size:]

# Convert pandas DataFrames to numpy arrays for training and testing
X_train = train_data[["receptions", "rec_yards", "rec_tds", "hof", "allpro", "probowls"]].values
y_train = train_data["round"].values
X_test = test_data[["receptions", "rec_yards", "rec_tds", "hof", "allpro", "probowls"]].values
y_test = test_data["round"].values

# Train a random forest classifier using our library
model = rice_ml.RandomForest(n_estimators=100, max_depth=3, min_samples_split=10)
model.train(X_train, y_train)

Now we have successfully trained our model. Note that this took significantly longer (12s) than the single decision tree, which makes sense due to the random forests' increased complexity. Now we shift to making predictions and evaluating model effectiveness.

In [ ]:
# Make predictions on the test set
predictions = model.predict(X_test)

# Evaluate the model's performance using accuracy and mean absolute error from our library
accuracy = rice_ml.accuracy_score(y_test, predictions)
mae = rice_ml.mean_absolute_error(y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print(f"Mean Absolute Error: {mae:.4f}")

Accuracy: 0.2288
Mean Absolute Error: 1.6340


In contrast with our hypothesis, the random forest model doesn't perform significantly better than the single tree. Because of the high variability of draft picks, the increased complexity likely overfits to the training data and therefore doesn't translate to test data as well as the simpler model. 

In this specific case, there isn't any true, strong trend so a simpler model is more effective in generalizing to test data. Our conclusion still lies in the fact that evaluating college players is difficult, so generalized models predciting overall career performance will always lack significant strength. 